# 00B_llava_onevision_7b_snapshot_smoke.ipynb

Generated zero-edit Kaggle runbook. Attach authenticated inputs under any account, dataset title, archive name, mount, or nesting; keep Internet off, choose the documented accelerator, and click Run All. NON_EVIDENCE_RUNTIME_SMOKE; paper_evidence=false.


In [ ]:
# Generated immutable run identity. There is nothing to edit in this notebook.
import os

STAGE = 'snapshot_smoke'
PROVIDER = 'llava_onevision_7b'
NOTEBOOK_NAME = '00B_llava_onevision_7b_snapshot_smoke.ipynb'
EXPECTED_GPUS = 0
ALLOW_SINGLE_GPU_FALLBACK = True
USE_REAL_MODEL = False
MAX_ITEMS = 2
GLOBAL_SEED = 12013
SCHEMA_VERSION = "certvic.cvpr.output.v2"
SNAPSHOT_CONTRACT = "UNIFIED_SNAPSHOT"
PROMPT_TEMPLATE_ID = "certification_yes_no_v1"
PROMPT_TEMPLATE = "{prompt}\n"
PARSER_VERSION = "certvic.parse.v2"
CANONICAL_RETURN_ZIP = '00B_llava_onevision_7b_snapshot_bundle.zip'
LOCAL_DESTINATION = 'data/runtime/00B_llava_onevision_7b_snapshot_bundle.zip'
WORKING_ROOT = os.environ.get("CERTVIC_KAGGLE_WORKING_ROOT", "/kaggle/working")
INPUT_ROOTS = [value for value in os.environ.get("CERTVIC_INPUT_ROOTS", "").split(os.pathsep)
               if value] or ["/kaggle/input", "/kaggle/working"]
for key, value in {
    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "DIFFUSERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1", "HF_HUB_DISABLE_TELEMETRY": "1", "PIP_NO_INDEX": "1",
    "PIP_DISABLE_PIP_VERSION_CHECK": "1",
}.items():
    os.environ[key] = value


In [ ]:
import hashlib, json, os, pathlib, shutil, stat, sys, zipfile

DISCOVERY_ERRORS = {
    "missing": "CERTVIC_DISCOVERY_01_REQUIRED_ROLE_NOT_FOUND",
    "ambiguous": "CERTVIC_DISCOVERY_02_AMBIGUOUS_DISTINCT_CONTENT",
    "authentication": "CERTVIC_DISCOVERY_03_CONTENT_AUTHENTICATION_FAILED",
}
DISCOVERY_POLICY = "CONTENT_AUTHENTICATED_ANY_LOCATION"
OPERATIONAL_FIELDS = {
    "builder_command", "created_time", "expected_kaggle_dataset_slug", "mount_path",
    "required_notebook", "validation_command",
}

def early_sha256(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def early_safe_member(info):
    name = info.filename
    normalized = name.replace("\\", "/")
    value = pathlib.PurePosixPath(normalized)
    mode = (info.external_attr >> 16) & 0xFFFF
    if (not normalized or normalized != name or normalized.endswith("/") or value.is_absolute()
            or ".." in value.parts or "." in value.parts or normalized.startswith("~")
            or "\x00" in normalized or info.is_dir() or stat.S_ISLNK(mode)
            or (mode and not stat.S_ISREG(mode))):
        raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: unsafe member {name!r}")
    return value.as_posix()

def early_content_identity(manifest, hash_files):
    identity_manifest = {key: value for key, value in manifest.items()
                         if key not in OPERATIONAL_FIELDS}
    identity_files = {name: record for name, record in hash_files.items()
                      if name not in {"README.md", "bundle_manifest.json"}}
    payload = json.dumps({"manifest": identity_manifest, "files": identity_files},
                         indent=2, sort_keys=True).encode() + b"\n"
    return hashlib.sha256(payload).hexdigest()

def early_verify_archive(path):
    with zipfile.ZipFile(path) as archive:
        infos = archive.infolist()
        names = [early_safe_member(info) for info in infos]
        if "bundle_manifest.json" not in names or "hash_manifest.json" not in names:
            return None
        if len(names) != len(set(names)) or archive.testzip() is not None:
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: duplicate or corrupt archive")
        manifest_bytes = archive.read("bundle_manifest.json")
        hash_bytes = archive.read("hash_manifest.json")
        if len(manifest_bytes) > 8 * 1024 * 1024 or len(hash_bytes) > 8 * 1024 * 1024:
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: oversized manifest")
        manifest = json.loads(manifest_bytes)
        hashes = json.loads(hash_bytes)
        if (manifest.get("schema") != "certvic.kaggle.bundle.v1"
                or hashes.get("schema") != "certvic.kaggle.hash_manifest.v1"
                or manifest.get("bundle_type") != "CODE"):
            return None
        declared, hash_files = manifest.get("files", {}), hashes.get("files", {})
        if (set(names) != set(hash_files) | {"hash_manifest.json"}
                or set(declared) != set(names) - {"bundle_manifest.json", "hash_manifest.json"}):
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: file universe mismatch")
        for name, record in hash_files.items():
            payload = archive.read(name)
            observed = {"size": len(payload), "sha256": hashlib.sha256(payload).hexdigest()}
            if record != observed or (name in declared and declared[name] != observed):
                raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: byte mismatch {name}")
        return manifest, hash_files, hashlib.sha256(manifest_bytes).hexdigest()

def early_verify_directory(path):
    root = pathlib.Path(path).resolve()
    manifest_path, hash_path = root / "bundle_manifest.json", root / "hash_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    hashes = json.loads(hash_path.read_text(encoding="utf-8"))
    if (manifest.get("schema") != "certvic.kaggle.bundle.v1"
            or hashes.get("schema") != "certvic.kaggle.hash_manifest.v1"
            or manifest.get("bundle_type") != "CODE"):
        return None
    observed = {}
    for member in root.rglob("*"):
        mode = member.lstat().st_mode
        if member.is_symlink() or not (stat.S_ISDIR(mode) or stat.S_ISREG(mode)):
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: unsafe extracted member")
        if stat.S_ISREG(mode):
            observed[member.relative_to(root).as_posix()] = member
    declared, hash_files = manifest.get("files", {}), hashes.get("files", {})
    if (set(observed) != set(hash_files) | {"hash_manifest.json"}
            or set(declared) != set(observed) - {"bundle_manifest.json", "hash_manifest.json"}):
        raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: extracted universe mismatch")
    for name, record in hash_files.items():
        member = observed[name]
        actual = {"size": member.stat().st_size, "sha256": early_sha256(member)}
        if actual != record or (name in declared and declared[name] != actual):
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: extracted byte mismatch {name}")
    return manifest, hash_files, early_sha256(manifest_path)

configured_roots = [value for value in os.environ.get("CERTVIC_INPUT_ROOTS", "").split(os.pathsep)
                    if value]
if not configured_roots:
    configured_roots = ["/kaggle/input", "/kaggle/working"]
INPUT_ROOTS = sorted({str(pathlib.Path(value).resolve()) for value in configured_roots
                      if pathlib.Path(value).is_dir() and not pathlib.Path(value).is_symlink()})
archive_candidates, directory_candidates = [], []
for root_value in INPUT_ROOTS:
    root = pathlib.Path(root_value)
    for current, directory_names, file_names in os.walk(root, followlinks=False):
        base = pathlib.Path(current)
        directory_names[:] = sorted(name for name in directory_names
                                     if not (base / name).is_symlink())
        if "bundle_manifest.json" in file_names and "hash_manifest.json" in file_names:
            directory_candidates.append(base.resolve())
        for name in sorted(file_names):
            candidate = base / name
            if candidate.is_symlink() or not candidate.is_file():
                continue
            try:
                with candidate.open("rb") as handle:
                    magic = handle.read(4)
            except OSError:
                continue
            if magic in {b"PK\x03\x04", b"PK\x05\x06", b"PK\x07\x08"}:
                archive_candidates.append(candidate.resolve())

valid, failures = [], []
for representation, candidates in (("zip_archive", sorted(set(archive_candidates))),
                                   ("extracted_directory", sorted(set(directory_candidates)))):
    for candidate in candidates:
        try:
            result = (early_verify_archive(candidate) if representation == "zip_archive"
                      else early_verify_directory(candidate))
        except (OSError, KeyError, json.JSONDecodeError, UnicodeDecodeError,
                zipfile.BadZipFile, RuntimeError) as error:
            failures.append(f"{candidate}: {error}")
            continue
        if result is None:
            continue
        manifest, hash_files, manifest_hash = result
        identity = early_content_identity(manifest, hash_files)
        expected = os.environ.get("CERTVIC_EXPECTED_CONTENT_ID_CODE")
        if expected and identity != expected.lower():
            failures.append(f"{candidate}: expected CODE content identity mismatch")
            continue
        valid.append({"path": candidate, "representation": representation,
                      "manifest": manifest, "manifest_sha256": manifest_hash,
                      "content_identity_sha256": identity})
if not valid:
    code = DISCOVERY_ERRORS["authentication"] if failures else DISCOVERY_ERRORS["missing"]
    raise RuntimeError(f"{code}: role=CODE failures={failures}")
identities = {row["content_identity_sha256"] for row in valid}
if len(identities) != 1:
    raise RuntimeError(f"{DISCOVERY_ERRORS['ambiguous']}: role=CODE candidates="
                       f"{[(row['content_identity_sha256'], str(row['path'])) for row in valid]}")
selected = min(valid, key=lambda row: os.path.normcase(str(row["path"])))
CODE_DISCOVERY_MIRRORS = sorted({str(row["path"]) for row in valid})
CODE_BUNDLE_SOURCE = str(selected["path"])
CODE_BUNDLE_HASH = selected["content_identity_sha256"]
CODE_ARCHIVE_SHA256 = (early_sha256(selected["path"])
                       if selected["representation"] == "zip_archive" else None)
if selected["representation"] == "zip_archive":
    CODE_EXTRACT_ROOT = pathlib.Path(os.environ.get(
        "CERTVIC_KAGGLE_WORKING_ROOT", "/kaggle/working")) / "certvic_code"
    if CODE_EXTRACT_ROOT.exists():
        if CODE_EXTRACT_ROOT.is_symlink() or not CODE_EXTRACT_ROOT.is_dir():
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: unsafe CODE destination")
        shutil.rmtree(CODE_EXTRACT_ROOT)
    CODE_EXTRACT_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(selected["path"]) as archive:
        for info in archive.infolist():
            name = early_safe_member(info)
            output = (CODE_EXTRACT_ROOT / name).resolve()
            output.relative_to(CODE_EXTRACT_ROOT.resolve())
            output.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as reader, output.open("xb") as writer:
                shutil.copyfileobj(reader, writer, length=1024 * 1024)
else:
    CODE_EXTRACT_ROOT = pathlib.Path(selected["path"])
CODE_BUNDLE_PATH = (CODE_BUNDLE_SOURCE if selected["representation"] == "zip_archive"
                    else str(CODE_EXTRACT_ROOT / "bundle_manifest.json"))
CODE_BUNDLE = CODE_BUNDLE_PATH
project_candidates = sorted(path.parent.resolve() for path in CODE_EXTRACT_ROOT.rglob("pyproject.toml")
                            if (path.parent / "certvic/__init__.py").is_file())
if len(project_candidates) != 1:
    raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: CODE project root ambiguous")
PROJECT_ROOT = project_candidates[0]
sys.path.insert(0, str(PROJECT_ROOT))
from certvic.cvpr.content_discovery import (
    DISCOVERY_POLICY, discover_authenticated_input, resolve_content_bound_roles,
)
from certvic.cvpr.notebook_bootstrap import discover_unique_file, discover_unique_root
authenticated_code = discover_authenticated_input(
    "CODE", roots=INPUT_ROOTS, expected_identity=CODE_BUNDLE_HASH,
    materialization_root=pathlib.Path(os.environ.get(
        "CERTVIC_KAGGLE_WORKING_ROOT", "/kaggle/working")) / "certvic_authenticated_inputs",
)
if authenticated_code["content_identity_sha256"] != CODE_BUNDLE_HASH:
    raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: early/shared CODE identity mismatch")
AUTHENTICATED_CONTENT_IDENTITIES = {"code_bundle": CODE_BUNDLE_HASH}
DISCOVERED_PROVENANCE = {"CODE": authenticated_code}
print({"discovery_policy": DISCOVERY_POLICY, "role": "CODE", "provider": None,
       "study": selected["manifest"].get("study"), "stage": selected["manifest"].get("stage"),
       "representation": selected["representation"], "discovered_path": CODE_BUNDLE_SOURCE,
       "content_identity_sha256": CODE_BUNDLE_HASH, "archive_sha256": CODE_ARCHIVE_SHA256,
       "mirrors": CODE_DISCOVERY_MIRRORS, "project_root": str(PROJECT_ROOT)})


In [ ]:
from certvic.cvpr.environment_lock import environment_lock_hash

DISCOVERY_MATERIALIZATION_ROOT = pathlib.Path(WORKING_ROOT) / "certvic_authenticated_inputs"
CONFIG_DATASET = discover_authenticated_input(
    "CONFIGS", roots=INPUT_ROOTS,
    materialization_root=DISCOVERY_MATERIALIZATION_ROOT,
)
TOOLS_DATASET = discover_authenticated_input(
    "EXECUTION_TOOLS", roots=INPUT_ROOTS,
    materialization_root=DISCOVERY_MATERIALIZATION_ROOT,
)
WHEELHOUSE_DATASET = discover_authenticated_input(
    "OFFLINE_LINUX_WHEELHOUSE", roots=INPUT_ROOTS,
    materialization_root=DISCOVERY_MATERIALIZATION_ROOT,
)
for discovered in (CONFIG_DATASET, TOOLS_DATASET, WHEELHOUSE_DATASET):
    DISCOVERED_PROVENANCE[discovered["role"]] = discovered
    print({key: discovered[key] for key in (
        "role", "provider", "study", "stage", "representation", "discovered_path",
        "materialized_root", "content_identity_sha256", "archive_sha256", "mirrors",
        "observed_mount", "observed_dataset_folder",
    )})
CONFIG_ROOT = pathlib.Path(CONFIG_DATASET["materialized_root"])
WHEELHOUSE_ROOT = pathlib.Path(WHEELHOUSE_DATASET["materialized_root"])
ENVIRONMENT_LOCK = str(discover_unique_file(CONFIG_ROOT, "kaggle_t4x2_environment.lock.json"))
ENVIRONMENT_LOCK_HASH = environment_lock_hash(ENVIRONMENT_LOCK)
WHEELHOUSE_MANIFEST = str(discover_unique_file(WHEELHOUSE_ROOT, "wheelhouse_manifest.json"))
WHEELHOUSE_PATH = str(WHEELHOUSE_ROOT / "wheels")
if not pathlib.Path(WHEELHOUSE_PATH).is_dir():
    raise RuntimeError("KAGGLE_BOOTSTRAP_04_WHEELHOUSE_INVALID: wheels directory missing")
MODEL_REGISTRY = str(discover_unique_file(CONFIG_ROOT, "certvic_immutable_model_registry.json"))
ATTACHED_INPUT_HASHES = {
    "code": CODE_BUNDLE_HASH,
    "configs": CONFIG_DATASET["content_identity_sha256"],
    "tools": TOOLS_DATASET["content_identity_sha256"],
    "wheelhouse": WHEELHOUSE_DATASET["content_identity_sha256"],
}
AUTHENTICATED_CONTENT_IDENTITIES.update({
    "configs": CONFIG_DATASET["content_identity_sha256"],
    "tools": TOOLS_DATASET["content_identity_sha256"],
    "wheelhouse": WHEELHOUSE_DATASET["content_identity_sha256"],
})
print({"environment_lock": ENVIRONMENT_LOCK, "environment_lock_hash": ENVIRONMENT_LOCK_HASH,
       "wheelhouse_manifest": WHEELHOUSE_MANIFEST,
       "authenticated_content_identities": AUTHENTICATED_CONTENT_IDENTITIES})


In [ ]:
from certvic.cvpr.model_snapshot_manifest import verify_manifest

SNAPSHOT_DATASET = discover_authenticated_input(
    "MODEL_SNAPSHOT", provider='llava_onevision_7b', stage="model_snapshot", roots=INPUT_ROOTS,
    materialization_root=DISCOVERY_MATERIALIZATION_ROOT,
)
DISCOVERED_PROVENANCE["MODEL_SNAPSHOT"] = SNAPSHOT_DATASET
print({key: SNAPSHOT_DATASET[key] for key in (
    "role", "provider", "study", "stage", "representation", "discovered_path",
    "materialized_root", "content_identity_sha256", "archive_sha256", "mirrors",
    "observed_mount", "observed_dataset_folder",
)})
SNAPSHOT_CONTAINER = pathlib.Path(SNAPSHOT_DATASET["materialized_root"])
SNAPSHOT_MANIFEST = str(discover_unique_file(
    SNAPSHOT_CONTAINER, "certvic_model_snapshot_manifest.json"
))
SNAPSHOT_ROOT = pathlib.Path(SNAPSHOT_MANIFEST).parent.resolve()
MODEL_PATH = str(SNAPSHOT_ROOT)
PROCESSOR_PATH = str(SNAPSHOT_ROOT)
SNAPSHOT_MANIFEST_HASH = early_sha256(SNAPSHOT_MANIFEST)
snapshot_identity = json.loads(pathlib.Path(SNAPSHOT_MANIFEST).read_text(encoding="utf-8"))
MODEL_ID = str(snapshot_identity["model_id"])
PROCESSOR_ID = str(snapshot_identity["processor_id"])
MODEL_COMMIT = str(snapshot_identity["model_commit"])
PROCESSOR_COMMIT = str(snapshot_identity["processor_commit"])
EXPECTED_ARCHITECTURE = str(snapshot_identity["expected_architecture"])
SNAPSHOT_ROOT_HASH = str(snapshot_identity["unified_snapshot_root_sha256"])
outer_snapshot = SNAPSHOT_DATASET["bundle_manifest"]
for field, expected in {
    "provider": PROVIDER, "model_id": MODEL_ID, "model_commit": MODEL_COMMIT,
    "processor_commit": PROCESSOR_COMMIT, "expected_architecture": EXPECTED_ARCHITECTURE,
    "unified_snapshot_root_sha256": SNAPSHOT_ROOT_HASH,
}.items():
    if outer_snapshot.get(field) != expected:
        raise RuntimeError(f"CERTVIC_DISCOVERY_03_CONTENT_AUTHENTICATION_FAILED: snapshot {field} mismatch")
registry = json.loads(pathlib.Path(MODEL_REGISTRY).read_text(encoding="utf-8"))["models"][PROVIDER]
if (registry.get("repository_id") != MODEL_ID
        or registry.get("model_commit") != MODEL_COMMIT
        or registry.get("processor_commit") != PROCESSOR_COMMIT
        or registry.get("architecture") != EXPECTED_ARCHITECTURE):
    raise RuntimeError("CERTVIC_DISCOVERY_03_CONTENT_AUTHENTICATION_FAILED: immutable registry mismatch")
ATTACHED_INPUT_HASHES["snapshot"] = SNAPSHOT_DATASET["content_identity_sha256"]
AUTHENTICATED_CONTENT_IDENTITIES["snapshot"] = SNAPSHOT_DATASET["content_identity_sha256"]
print({"snapshot_root": MODEL_PATH, "snapshot_manifest": SNAPSHOT_MANIFEST,
       "snapshot_manifest_sha256": SNAPSHOT_MANIFEST_HASH,
       "snapshot_root_sha256": SNAPSHOT_ROOT_HASH, "model_id": MODEL_ID,
       "model_commit": MODEL_COMMIT, "processor_commit": PROCESSOR_COMMIT,
       "expected_architecture": EXPECTED_ARCHITECTURE})


In [ ]:
from certvic.cvpr.environment_lock import (
    offline_environment_flags, prepare_offline_environment,
)
from certvic.cvpr.notebook_bootstrap import configure_offline_environment, import_smoke
from certvic.cvpr.runtime_preflight import hardware_report

configure_offline_environment()
if offline_environment_flags().get("HF_HUB_OFFLINE") != "1" or os.environ.get("PIP_NO_INDEX") != "1":
    raise RuntimeError("KAGGLE_ZERO_EDIT_OFFLINE_POLICY_INCOMPLETE")
environment_verification = prepare_offline_environment(
    ENVIRONMENT_LOCK,
    wheelhouse=WHEELHOUSE_PATH,
    wheelhouse_manifest=WHEELHOUSE_MANIFEST,
    allow_preinstalled=True,
    require_exact=True,
    require_cuda=False,
)
if environment_verification["status"] not in {
    "EXACT_PREINSTALLED_ENVIRONMENT_ACCEPTED", "OFFLINE_WHEELHOUSE_INSTALLED_AND_VERIFIED",
}:
    raise RuntimeError("KAGGLE_ZERO_EDIT_EXACT_ENVIRONMENT_NOT_ESTABLISHED")
hardware = hardware_report()
print(hardware)
if EXPECTED_GPUS == 0 and (hardware["cuda_available"] or hardware["gpu_count"] != 0):
    raise RuntimeError("KAGGLE_ZERO_EDIT_CPU_ACCELERATOR_MUST_BE_OFF")
if EXPECTED_GPUS > 0:
    names = [row["name"] for row in hardware.get("gpus", [])]
    if not hardware["cuda_available"]:
        raise RuntimeError("KAGGLE_BOOTSTRAP_07_GPU_CONTRACT_FAILED: CUDA unavailable")
    if len(names) < 2 and not (len(names) == 1 and ALLOW_SINGLE_GPU_FALLBACK):
        raise RuntimeError(f"KAGGLE_BOOTSTRAP_07_GPU_CONTRACT_FAILED: device_count={len(names)}")
    if not all("T4" in name.upper() for name in names[:2]):
        raise RuntimeError(f"KAGGLE_BOOTSTRAP_07_GPU_CONTRACT_FAILED: devices={names}")
import_versions = import_smoke(["certvic", "numpy", "pandas", "torch", "transformers"])
print({"offline": True, "network_used": False, "imports": import_versions,
       "environment_status": environment_verification["status"]})


In [ ]:
from certvic.cvpr.smoke_artifacts import write_snapshot_artifacts

snapshot = verify_manifest(
    MODEL_PATH, SNAPSHOT_MANIFEST,
    expected_model_id=MODEL_ID,
    expected_model_commit=MODEL_COMMIT,
    expected_processor_commit=PROCESSOR_COMMIT,
    expected_architecture=EXPECTED_ARCHITECTURE,
)
if not snapshot["passed"]:
    raise RuntimeError(f"KAGGLE_ZERO_EDIT_SNAPSHOT_INVALID: {snapshot['errors']}")
artifact = write_snapshot_artifacts(WORKING_ROOT, PROVIDER, {
    **snapshot,
    "snapshot_contract": SNAPSHOT_CONTRACT,
    "model_id": MODEL_ID,
    "processor_id": PROCESSOR_ID,
    "model_commit": MODEL_COMMIT,
    "processor_commit": PROCESSOR_COMMIT,
    "expected_architecture": EXPECTED_ARCHITECTURE,
    "snapshot_manifest_file_sha256": SNAPSHOT_MANIFEST_HASH,
    "snapshot_root_hash": SNAPSHOT_ROOT_HASH,
    "snapshot_archive_sha256": SNAPSHOT_DATASET.get("archive_sha256"),
    "snapshot_content_identity_sha256": SNAPSHOT_DATASET["content_identity_sha256"],
})
canonical = pathlib.Path(WORKING_ROOT) / CANONICAL_RETURN_ZIP
if not canonical.is_file():
    raise RuntimeError(f"KAGGLE_ZERO_EDIT_CANONICAL_RETURN_MISSING: {CANONICAL_RETURN_ZIP}")
print(str(canonical))
print(f"DOWNLOAD_FILENAME={CANONICAL_RETURN_ZIP}")
print(f"LOCAL_DESTINATION={LOCAL_DESTINATION}")
print("RESUME_COMMAND=python3 scripts/run_all_cpu_workflows.py --resume")
